# MediFlow Hair — 중복·충돌 제거 데이터셋 생성기

이 노트북은 기존 `hair_processed.zip`을 읽어 정리된 새 데이터셋을 만듭니다. 원본 ZIP과 기존 결과는 수정하거나 삭제하지 않습니다.

정리 규칙:

1. 파일 내용이 같지만 클래스가 다르면 해당 이미지 그룹 전체를 제외합니다. 정답을 자동으로 추측하지 않습니다.
2. 파일 내용과 클래스가 모두 같은 중복은 한 장만 남깁니다.
3. 남은 고유 이미지를 클래스별로 Train 80%, Validation 10%, Test 10%로 새로 나눕니다.
4. 증강은 새 Train에서만 다시 만들고, Validation과 Test는 원본을 그대로 복사합니다.
5. 모든 제외·선택·분할·증강 기록을 CSV와 JSON으로 저장합니다.

> 사람·병변·촬영 세션 식별 정보가 없으므로 동일 인물이나 같은 세션의 다른 사진까지 분리됐는지는 확인할 수 없습니다. 이 한계는 결과 기록에 남습니다.


## 1. Colab과 Drive 준비

GPU는 필요하지 않습니다. 이 노트북은 데이터만 정리하며 학습하지 않습니다. Drive 연결 창이 나타나면 본인의 계정을 승인합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import platform
import random
import shutil
import zipfile
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from tqdm.auto import tqdm

print('Python:', platform.python_version())


## 2. 설정

Drive 최상위와 기존 `hair_dataset` 폴더를 자동으로 찾습니다. 실패할 때만 `DATA_ZIP_OVERRIDE`에 정확한 경로를 입력합니다. 출력 이름에는 실행 시각이 포함되므로 기존 파일을 덮어쓰지 않습니다.


In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ZIP_OVERRIDE = ''
CLASS_NAMES = ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다']
SPLIT_NAMES = ('train', 'val', 'test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SEED = 42
AUGMENTATION_RATIO = 0.5
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
DATASET_NAME = f'hair_clean_v1_{RUN_ID}'
LOCAL_ROOT = Path('/content') / DATASET_NAME
SOURCE_ZIP_LOCAL = Path('/content') / f'hair_source_{RUN_ID}.zip'
EXTRACT_ROOT = Path('/content') / f'hair_source_{RUN_ID}'
DATASET_ROOT = LOCAL_ROOT / DATASET_NAME
REPORT_ROOT = DATASET_ROOT / 'reports'
DRIVE_OUTPUT_ROOT = MY_DRIVE / 'mediflow_datasets'
DRIVE_ZIP_PATH = DRIVE_OUTPUT_ROOT / f'{DATASET_NAME}.zip'
DRIVE_REPORT_DIR = DRIVE_OUTPUT_ROOT / f'{DATASET_NAME}_reports'

random.seed(SEED)
np.random.seed(SEED)
if LOCAL_ROOT.exists() or EXTRACT_ROOT.exists():
    raise FileExistsError('이번 실행의 임시 폴더가 이미 존재합니다. RUN_ID를 새로 만든 뒤 다시 실행하세요.')
DATASET_ROOT.mkdir(parents=True)
REPORT_ROOT.mkdir()
print('새 데이터셋 이름:', DATASET_NAME)
print('최종 Drive ZIP:', DRIVE_ZIP_PATH)


## 3. 원본 ZIP 찾기·복사·압축 해제

원본 ZIP을 Colab 임시 공간으로 복사하면서 SHA-256 식별값을 계산합니다. ZIP 내부 경로도 검사한 뒤 안전하게 압축을 풉니다.


In [ ]:
def find_data_zip():
    if DATA_ZIP_OVERRIDE:
        candidates = [Path(DATA_ZIP_OVERRIDE)]
    else:
        direct = [
            MY_DRIVE / 'hair_processed.zip',
            MY_DRIVE / 'hair_processed',
            MY_DRIVE / 'hair_dataset' / 'hair_processed.zip',
        ]
        candidates = [path for path in direct if path.is_file() and zipfile.is_zipfile(path)]
        if not candidates:
            candidates = [
                path for path in MY_DRIVE.rglob('hair_processed*')
                if path.is_file() and zipfile.is_zipfile(path)
            ]
    candidates = [path for path in candidates if path.is_file() and zipfile.is_zipfile(path)]
    if not candidates:
        raise FileNotFoundError('Drive에서 hair_processed ZIP을 찾지 못했습니다.')
    if len(candidates) > 1:
        raise ValueError(f'데이터 ZIP 후보가 여러 개입니다. DATA_ZIP_OVERRIDE를 지정하세요: {candidates}')
    return candidates[0]

def copy_with_sha256(source, destination):
    digest = hashlib.sha256()
    with source.open('rb') as src, destination.open('wb') as dst:
        while True:
            chunk = src.read(8 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
            dst.write(chunk)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=False)
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination_resolved / member.filename).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
        archive.extractall(destination_resolved)

SOURCE_ZIP_PATH = find_data_zip()
print('원본 ZIP:', SOURCE_ZIP_PATH)
SOURCE_ZIP_SHA256 = copy_with_sha256(SOURCE_ZIP_PATH, SOURCE_ZIP_LOCAL)
print('원본 ZIP SHA-256:', SOURCE_ZIP_SHA256)
safe_extract(SOURCE_ZIP_LOCAL, EXTRACT_ROOT)
print('압축 해제 완료:', EXTRACT_ROOT)


## 4. Original 이미지 전체 조사

기존 Augmented 이미지는 사용하지 않습니다. 어느 원본에서 증강됐는지 연결 기록이 없기 때문입니다. Original의 모든 분할을 합쳐 파일 내용과 라벨을 다시 검사합니다.


In [ ]:
def find_original_root():
    matches = [
        path for path in EXTRACT_ROOT.rglob('original')
        if path.is_dir() and all((path / split).is_dir() for split in SPLIT_NAMES)
    ]
    if len(matches) != 1:
        raise ValueError(f'original 폴더를 하나로 확정할 수 없습니다: {matches}')
    return matches[0]

def image_files(folder):
    return sorted(
        path for path in folder.rglob('*')
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

ORIGINAL_ROOT = find_original_root()
found_classes = sorted(path.name for path in (ORIGINAL_ROOT / 'train').iterdir() if path.is_dir())
if found_classes != sorted(CLASS_NAMES):
    raise ValueError(f'클래스 구성이 예상과 다릅니다: {found_classes}')

inventory_rows = []
for split in SPLIT_NAMES:
    for class_name in CLASS_NAMES:
        files = image_files(ORIGINAL_ROOT / split / class_name)
        if not files:
            raise ValueError(f'{split}/{class_name} 이미지가 없습니다.')
        for path in tqdm(files, desc=f'{split}/{class_name} 해시 검사'):
            inventory_rows.append({
                'source_split': split,
                'class_name': class_name,
                'source_path': str(path.relative_to(ORIGINAL_ROOT)),
                'absolute_path': str(path),
                'sha256': sha256_file(path),
                'size_bytes': path.stat().st_size,
            })
inventory_df = pd.DataFrame(inventory_rows)
inventory_df.drop(columns=['absolute_path']).to_csv(
    REPORT_ROOT / 'source_inventory.csv', index=False, encoding='utf-8-sig'
)
print('Original 파일 수:', len(inventory_df))
display(pd.crosstab(inventory_df['class_name'], inventory_df['source_split']))


## 5. 충돌 제거와 고유 이미지 선택

같은 SHA-256 그룹에 클래스가 둘 이상이면 모두 제외합니다. 같은 클래스의 중복은 경로 정렬상 첫 번째 파일 한 장만 대표로 사용합니다. 제외된 이유와 경로는 보고서로 남깁니다.


In [ ]:
conflict_rows = []
duplicate_rows = []
selected_rows = []
for digest, group in inventory_df.groupby('sha256', sort=True):
    labels = sorted(group['class_name'].unique().tolist())
    ordered = group.sort_values(['class_name', 'source_split', 'source_path'])
    if len(labels) > 1:
        for _, row in ordered.iterrows():
            conflict_rows.append({
                'sha256': digest,
                'conflicting_labels': '|'.join(labels),
                'class_name': row['class_name'],
                'source_split': row['source_split'],
                'source_path': row['source_path'],
                'reason': '같은 파일 내용에 서로 다른 클래스 라벨',
            })
        continue
    representative = ordered.iloc[0].to_dict()
    selected_rows.append(representative)
    for _, row in ordered.iloc[1:].iterrows():
        duplicate_rows.append({
            'sha256': digest,
            'class_name': row['class_name'],
            'kept_source_path': representative['source_path'],
            'removed_source_path': row['source_path'],
            'reason': '같은 클래스의 완전 동일 파일 중복',
        })

conflict_df = pd.DataFrame(conflict_rows)
duplicate_df = pd.DataFrame(duplicate_rows)
selected_df = pd.DataFrame(selected_rows)
conflict_df.to_csv(REPORT_ROOT / 'removed_label_conflicts.csv', index=False, encoding='utf-8-sig')
duplicate_df.to_csv(REPORT_ROOT / 'removed_same_label_duplicates.csv', index=False, encoding='utf-8-sig')
print('서로 다른 라벨 때문에 제외한 파일 수:', len(conflict_df))
print('같은 라벨 중복으로 제외한 파일 수:', len(duplicate_df))
print('남은 고유·단일 라벨 이미지 수:', len(selected_df))
display(selected_df.groupby('class_name').size().rename('usable_images').to_frame())
if selected_df.empty or set(selected_df['class_name']) != set(CLASS_NAMES):
    raise ValueError('정리 후 일부 클래스에 사용할 이미지가 없습니다.')


## 6. 새 80:10:10 분할 생성

클래스마다 SHA-256과 Seed를 사용해 순서를 고정합니다. 같은 입력 ZIP과 Seed로 실행하면 같은 분할이 만들어집니다. 파일은 재압축하거나 내용을 변경하지 않고 복사합니다.


In [ ]:
for dataset_type in ('original', 'augmented'):
    for split in SPLIT_NAMES:
        for class_name in CLASS_NAMES:
            (DATASET_ROOT / dataset_type / split / class_name).mkdir(parents=True)

split_rows = []
for class_name in CLASS_NAMES:
    class_rows = selected_df[selected_df['class_name'] == class_name].copy()
    class_rows['split_key'] = class_rows['sha256'].map(
        lambda digest: hashlib.sha256(f'{SEED}:{class_name}:{digest}'.encode('utf-8')).hexdigest()
    )
    class_rows = class_rows.sort_values('split_key').reset_index(drop=True)
    total = len(class_rows)
    test_count = total // 10
    val_count = total // 10
    if test_count == 0 or val_count == 0:
        raise ValueError(f'{class_name}의 이미지가 분할하기에 부족합니다: {total}')
    assignments = ['test'] * test_count + ['val'] * val_count + ['train'] * (total - test_count - val_count)
    class_rows['new_split'] = assignments
    counters = defaultdict(int)
    for _, row in tqdm(class_rows.iterrows(), total=total, desc=f'{class_name} 새 분할'):
        split = row['new_split']
        counters[split] += 1
        source_path = Path(row['absolute_path'])
        suffix = source_path.suffix.lower()
        output_name = f'{class_name}_{counters[split]:05d}{suffix}'
        output_path = DATASET_ROOT / 'original' / split / class_name / output_name
        shutil.copy2(source_path, output_path)
        split_rows.append({
            'sha256': row['sha256'],
            'class_name': class_name,
            'old_split': row['source_split'],
            'source_path': row['source_path'],
            'new_split': split,
            'output_path': str(output_path.relative_to(DATASET_ROOT)),
        })
split_df = pd.DataFrame(split_rows)
split_df.to_csv(REPORT_ROOT / 'split_manifest.csv', index=False, encoding='utf-8-sig')
split_counts = pd.crosstab(split_df['class_name'], split_df['new_split']).reindex(
    index=CLASS_NAMES, columns=['train', 'val', 'test'], fill_value=0
)
display(split_counts)


## 7. Augmented 데이터 생성

새 Original을 Augmented 구조에 복사한 뒤 Train에만 원본 수의 50%만큼 새 증강 이미지를 추가합니다. 각 증강 파일이 어느 Train 원본에서 만들어졌는지 기록합니다.


In [ ]:
for split in SPLIT_NAMES:
    for class_name in CLASS_NAMES:
        source_dir = DATASET_ROOT / 'original' / split / class_name
        destination_dir = DATASET_ROOT / 'augmented' / split / class_name
        for source_path in image_files(source_dir):
            shutil.copy2(source_path, destination_dir / source_path.name)

def augment_image(image, rng, np_rng):
    image = image.convert('RGB')
    operations = []
    angle = rng.uniform(-8, 8)
    image = image.rotate(angle, resample=Image.Resampling.BILINEAR)
    operations.append({'rotation_degrees': angle})
    if rng.random() < 0.25:
        image = ImageOps.mirror(image)
        operations.append({'horizontal_flip': True})
    brightness = rng.uniform(0.82, 1.18)
    image = ImageEnhance.Brightness(image).enhance(brightness)
    operations.append({'brightness': brightness})
    if rng.random() < 0.65:
        contrast = rng.uniform(0.85, 1.15)
        image = ImageEnhance.Contrast(image).enhance(contrast)
        operations.append({'contrast': contrast})
    if rng.random() < 0.35:
        color = rng.uniform(0.90, 1.10)
        image = ImageEnhance.Color(image).enhance(color)
        operations.append({'color': color})
    if rng.random() < 0.25:
        radius = rng.uniform(0.15, 0.45)
        image = image.filter(ImageFilter.GaussianBlur(radius))
        operations.append({'gaussian_blur_radius': radius})
    if rng.random() < 0.25:
        std = rng.uniform(1.5, 4.0)
        array = np.asarray(image).astype(np.float32)
        array += np_rng.normal(0, std, array.shape)
        image = Image.fromarray(np.clip(array, 0, 255).astype(np.uint8))
        operations.append({'gaussian_noise_std': std})
    return image, operations

augmentation_rows = []
for class_index, class_name in enumerate(CLASS_NAMES):
    train_sources = image_files(DATASET_ROOT / 'original' / 'train' / class_name)
    augmentation_count = int(len(train_sources) * AUGMENTATION_RATIO)
    chooser = random.Random(SEED + class_index)
    selected_sources = chooser.sample(train_sources, augmentation_count)
    destination_dir = DATASET_ROOT / 'augmented' / 'train' / class_name
    for index, source_path in enumerate(tqdm(selected_sources, desc=f'{class_name} 증강'), start=1):
        item_seed = SEED * 100000 + class_index * 10000 + index
        rng = random.Random(item_seed)
        np_rng = np.random.default_rng(item_seed)
        with Image.open(source_path) as source_image:
            augmented, operations = augment_image(source_image, rng, np_rng)
        output_path = destination_dir / f'{class_name}_aug_{index:05d}.jpg'
        augmented.save(output_path, 'JPEG', quality=95, optimize=True)
        augmentation_rows.append({
            'class_name': class_name,
            'source_path': str(source_path.relative_to(DATASET_ROOT)),
            'output_path': str(output_path.relative_to(DATASET_ROOT)),
            'seed': item_seed,
            'operations': json.dumps(operations, ensure_ascii=False),
        })
augmentation_df = pd.DataFrame(augmentation_rows)
augmentation_df.to_csv(REPORT_ROOT / 'augmentation_manifest.csv', index=False, encoding='utf-8-sig')
print('새 증강 이미지 수:', len(augmentation_df))


## 8. 최종 검증

새 Original 분할 사이의 완전 동일 파일이 없는지 다시 확인합니다. Augmented Validation/Test가 Original과 완전히 같은지도 확인합니다.


In [ ]:
def hash_set(folder):
    return {sha256_file(path) for path in image_files(folder)}

clean_hashes = {split: hash_set(DATASET_ROOT / 'original' / split) for split in SPLIT_NAMES}
final_overlaps = {
    'train_val': len(clean_hashes['train'] & clean_hashes['val']),
    'train_test': len(clean_hashes['train'] & clean_hashes['test']),
    'val_test': len(clean_hashes['val'] & clean_hashes['test']),
}
if any(final_overlaps.values()):
    raise ValueError(f'새 분할에도 중복이 있습니다: {final_overlaps}')
for split in ('val', 'test'):
    if hash_set(DATASET_ROOT / 'original' / split) != hash_set(DATASET_ROOT / 'augmented' / split):
        raise ValueError(f'Augmented/{split}가 Original/{split}와 다릅니다.')

final_counts = {}
for dataset_type in ('original', 'augmented'):
    final_counts[dataset_type] = {}
    for split in SPLIT_NAMES:
        final_counts[dataset_type][split] = {
            class_name: len(image_files(DATASET_ROOT / dataset_type / split / class_name))
            for class_name in CLASS_NAMES
        }
summary = {
    'dataset_name': DATASET_NAME,
    'created_at': datetime.now().isoformat(),
    'source_zip_path': str(SOURCE_ZIP_PATH),
    'source_zip_sha256': SOURCE_ZIP_SHA256,
    'classes': CLASS_NAMES,
    'seed': SEED,
    'split_rule': 'class별 80:10:10, SHA-256 기반 결정적 순서',
    'augmentation_ratio': AUGMENTATION_RATIO,
    'source_original_file_count': len(inventory_df),
    'removed_conflicting_file_count': len(conflict_df),
    'removed_same_label_duplicate_file_count': len(duplicate_df),
    'usable_unique_file_count': len(selected_df),
    'final_counts': final_counts,
    'exact_duplicate_overlap_after_cleaning': final_overlaps,
    'limitations': [
        '사람·병변·촬영 세션 식별 정보가 없어 해당 단위의 분할 누수는 확인하지 못함',
        '내용이 약간 변경된 유사 이미지나 재인코딩 이미지는 SHA-256 검사만으로 찾지 못함',
        '서로 다른 픽셀의 이미지에 잘못 붙은 라벨은 이번 검사만으로 확인하지 못함',
    ],
}
(REPORT_ROOT / 'dataset_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('최종 검증 통과:', final_overlaps)
display(pd.DataFrame(final_counts['original']))
display(pd.DataFrame(final_counts['augmented']))


## 9. ZIP 생성 및 Drive 저장

정리된 데이터셋 전체를 ZIP 하나로 만들고 Drive에 복사합니다. 복사가 끝난 뒤 원본과 Drive 파일의 크기 및 SHA-256을 비교합니다. 기존 파일이 있으면 덮어쓰지 않고 중단합니다.


In [ ]:
LOCAL_ZIP_PATH = Path(shutil.make_archive(
    str(LOCAL_ROOT / DATASET_NAME),
    'zip',
    root_dir=DATASET_ROOT.parent,
    base_dir=DATASET_ROOT.name,
))
LOCAL_ZIP_SHA256 = sha256_file(LOCAL_ZIP_PATH)
print('새 ZIP:', LOCAL_ZIP_PATH)
print('새 ZIP 크기:', f'{LOCAL_ZIP_PATH.stat().st_size / (1024 ** 3):.2f} GB')
print('새 ZIP SHA-256:', LOCAL_ZIP_SHA256)

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if DRIVE_ZIP_PATH.exists() or DRIVE_REPORT_DIR.exists():
    raise FileExistsError('같은 이름의 Drive 출력이 있습니다. 기존 파일은 덮어쓰지 않습니다.')
partial_path = DRIVE_ZIP_PATH.with_suffix('.zip.partial')
if partial_path.exists():
    raise FileExistsError(f'이전의 미완료 복사 파일이 있습니다: {partial_path}')
shutil.copy2(LOCAL_ZIP_PATH, partial_path)
if partial_path.stat().st_size != LOCAL_ZIP_PATH.stat().st_size:
    raise IOError('Drive 복사 후 파일 크기가 다릅니다.')
DRIVE_COPY_SHA256 = sha256_file(partial_path)
if DRIVE_COPY_SHA256 != LOCAL_ZIP_SHA256:
    raise IOError('Drive 복사 후 SHA-256이 다릅니다.')
partial_path.replace(DRIVE_ZIP_PATH)
shutil.copytree(REPORT_ROOT, DRIVE_REPORT_DIR)

final_record = {
    'drive_zip_path': str(DRIVE_ZIP_PATH),
    'drive_report_dir': str(DRIVE_REPORT_DIR),
    'zip_size_bytes': DRIVE_ZIP_PATH.stat().st_size,
    'zip_sha256': DRIVE_COPY_SHA256,
}
(DRIVE_REPORT_DIR / 'drive_output.json').write_text(
    json.dumps(final_record, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Drive 저장 완료')
print('데이터 ZIP:', DRIVE_ZIP_PATH)
print('검사 보고서:', DRIVE_REPORT_DIR)


## 완료 후

마지막 셀에 `Drive 저장 완료`가 표시되어야 생성이 끝난 것입니다. 새 ZIP 내부에는 `original`, `augmented`, `reports`가 있으며 기존 `hair_processed.zip`은 그대로 남습니다. 이 데이터셋은 정확한 파일 중복과 확인된 라벨 충돌을 제거했지만, 사람·촬영 세션 단위의 누수까지 검증됐다는 뜻은 아닙니다.
